# Seq2Seq LSTM (Encoder–Decoder) using Keras

This notebook demonstrates a simple character-level Seq2Seq LSTM model for machine translation.

**Dataset:** English–French sentence pairs (`fra-eng`) from the ManyThings project.

Pipeline:
1. Download dataset
2. Preprocess text
3. One-hot encode characters
4. Build Encoder
5. Build Decoder
6. Train model
7. Build inference models
8. Test translations


## Dataset

This notebook uses the **ManyThings English–French Translation Dataset**.

The dataset (`fra.txt`) contains tab-separated sentence pairs:

```
Go.        Va !
Hi.        Salut !
Thank you. Merci.
```

Each row:
```
English<TAB>French
```


In [ ]:
import os
import zipfile
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, LSTM, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.utils import get_file

In [ ]:
# Download dataset
path = get_file(
    "fra-eng.zip",
    origin="https://www.manythings.org/anki/fra-eng.zip",
    extract=True
)

dataset_dir = os.path.join(os.path.dirname(path), "fra-eng")
data_path = os.path.join(dataset_dir, "fra.txt")
print(data_path)

### Preprocessing
Fill this section using the preprocessing from the Keras example (character extraction, one-hot encoding, encoder_input_data, decoder_input_data, decoder_target_data).

In [ ]:
# Hyperparameters
latent_dim = 256
batch_size = 64
epochs = 20

# Assume the following variables are created during preprocessing:
# encoder_input_data
# decoder_input_data
# decoder_target_data
# num_encoder_tokens
# num_decoder_tokens
# target_token_index
# reverse_target_char_index
# max_encoder_seq_length
# input_token_index
# input_texts


In [ ]:
# Encoder
encoder_inputs = Input(shape=(None, num_encoder_tokens))
encoder_lstm = LSTM(latent_dim, return_state=True)
encoder_outputs, state_h, state_c = encoder_lstm(encoder_inputs)
encoder_states = [state_h, state_c]

# Decoder
decoder_inputs = Input(shape=(None, num_decoder_tokens))
decoder_lstm = LSTM(latent_dim,
                    return_sequences=True,
                    return_state=True)

decoder_outputs, _, _ = decoder_lstm(
    decoder_inputs,
    initial_state=encoder_states
)

decoder_dense = Dense(num_decoder_tokens, activation="softmax")
decoder_outputs = decoder_dense(decoder_outputs)

model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer="adam",
              loss="categorical_crossentropy",
              metrics=["accuracy"])

model.summary()

In [ ]:
# Train
history = model.fit(
    [encoder_input_data, decoder_input_data],
    decoder_target_data,
    batch_size=batch_size,
    epochs=epochs,
    validation_split=0.2
)

In [ ]:
# Encoder inference
encoder_model = Model(
    encoder_inputs,
    encoder_states
)

# Decoder inference
decoder_state_input_h = Input(shape=(latent_dim,))
decoder_state_input_c = Input(shape=(latent_dim,))
decoder_states_inputs = [
    decoder_state_input_h,
    decoder_state_input_c
]

decoder_outputs, state_h, state_c = decoder_lstm(
    decoder_inputs,
    initial_state=decoder_states_inputs
)

decoder_states = [state_h, state_c]
decoder_outputs = decoder_dense(decoder_outputs)

decoder_model = Model(
    [decoder_inputs] + decoder_states_inputs,
    [decoder_outputs] + decoder_states
)

In [ ]:
def decode_sequence(input_seq):

    states = encoder_model.predict(input_seq, verbose=0)

    target_seq = np.zeros((1,1,num_decoder_tokens))
    target_seq[0,0,target_token_index["\t"]] = 1

    decoded_sentence = ""

    while True:

        output_tokens, h, c = decoder_model.predict(
            [target_seq] + states,
            verbose=0
        )

        sampled_index = np.argmax(output_tokens[0,-1,:])
        sampled_char = reverse_target_char_index[sampled_index]

        decoded_sentence += sampled_char

        if sampled_char == "\n" or len(decoded_sentence) > 50:
            break

        target_seq = np.zeros((1,1,num_decoder_tokens))
        target_seq[0,0,sampled_index] = 1
        states = [h,c]

    return decoded_sentence.strip()

In [ ]:
# Test on training samples
for i in range(5):
    pred = decode_sequence(encoder_input_data[i:i+1])
    print("Input :", input_texts[i])
    print("Output:", pred)
    print("-"*40)